# Data Preprocessing — AssureX Python Classification Model

This notebook documents the preprocessing steps used before training the
Python tabular classifier: feature engineering, missing-value handling,
categorical encoding and numerical scaling.

**Target:** classify a claim as `Valid`, `Invalid` or `Manual_Review`.

## 1. Load the structured dataset

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.read_csv('dataset/train.csv')
val = pd.read_csv('dataset/val.csv')
test = pd.read_csv('dataset/test.csv')
print('train:', train.shape, '| val:', val.shape, '| test:', test.shape)
print(train['class_label'].value_counts())

## 2. Feature definition

13 numeric features + 2 categorical features.

In [ ]:
NUMERIC_FEATURES = [
    'product_age_months', 'warranty_duration_months', 'remaining_warranty_months',
    'purchase_price', 'repair_history_count', 'unauthorized_repair_flag',
    'serial_number_match', 'receipt_attached', 'warranty_card_attached',
    'damage_photo_attached', 'serial_photo_attached', 'contradiction_flag',
    'duplicate_flag',
]
CATEGORICAL_FEATURES = ['product_category', 'fault_type']
TARGET = 'class_label'

## 3. Missing-value handling

Numeric fields default to `0`; categorical fields get safe defaults
(`Electronics`, `other`). This keeps the feature vector complete.

In [ ]:
def handle_missing(df):
    df = df.copy()
    for c in NUMERIC_FEATURES:
        df[c] = df[c].fillna(0)
    df['product_category'] = df['product_category'].fillna('Electronics')
    df['fault_type'] = df['fault_type'].fillna('other')
    return df

train = handle_missing(train)
print('nulls after handling:', int(train[NUMERIC_FEATURES + CATEGORICAL_FEATURES].isna().sum().sum()))

## 4. Encoding + scaling pipeline

- `StandardScaler` for the 13 numeric features
- `OneHotEncoder(handle_unknown='ignore')` for the 2 categorical features

In [ ]:
pre = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ],
    remainder='drop',
)

X = train[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
Xt = pre.fit_transform(X)
print('input features :', X.shape[1])
print('output features:', Xt.shape[1])
list(pre.get_feature_names_out())

## 5. Full pipeline (preprocessing + classifier)

The preprocessing is bundled with the classifier so that training and serving
apply identical transformations. This pipeline is what gets saved to
`model/python_claim_model.pkl`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([
    ('pre', pre),
    ('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)),
])
pipe.fit(X, train[TARGET])
print('pipeline trained on', X.shape[0], 'rows')